In [12]:
#1. Install and configure Gemini and Gradio
!pip install -q -U google-genai gradio
from google import genai
from getpass import getpass
import os
import gradio as gr

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")
client = genai.Client()


Enter your Gemini API Key: ··········


In [13]:
#2. Give sample student inputs

sample_previous_log = """
Complete Python lab record - High priority - unfinished
Revise Machine Learning notes - Medium priority - completed
Practice 5 aptitude problems - Medium priority - unfinished
Prepare for internal assessment - High priority - unfinished
"""
sample_today_goals = """
Complete Agentic AI lab program - High priority
Submit pending assignment - High priority
Practice one coding problem - Medium priority
Revise one placement topic - Medium priority
"""
sample_time_slots = """
09:00 AM to 10:30 AM
02:00 PM to 03:30 PM
07:00 PM to 08:00 PM
"""


In [14]:
#3. Create the LLM-based Daily Planner Agent
class DailyPlannerAgent:
    state = {}
    logs = []

    def plan_day(self, previous_log, today_goals, time_slots):
        self.state = {
            "previous_log": previous_log,
            "today_goals": today_goals,
            "time_slots": time_slots
        }

        prompt = f"""
You are a self-reflective daily planner agent for an engineering student.
Previous day log:
{previous_log}

Today's new goals:
{today_goals}

Available time slots:
{time_slots}

Do the following:
1. Reflect on unfinished tasks from the previous day.
2. Give importance to unfinished and high-priority tasks.
3. Re-plan the day using the available time slots.
4. Mention tasks that cannot be completed today as pending.
5. Give a short summary and self-check.

Use these headings:
Reflection
Today's Re-Planned Schedule
Pending Tasks
Summary
Self-Check
"""

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        final_plan = response.text
        self.state["final_plan"] = final_plan
        self.logs.append(self.state.copy())
        return final_plan



#4: Create and launch the Gradio app
agent = DailyPlannerAgent()
def daily_planner_app(previous_log, today_goals, time_slots):
    return agent.plan_day(previous_log, today_goals, time_slots)

app = gr.Interface(
    fn=daily_planner_app,
    inputs=[
        gr.Textbox(value=sample_previous_log, label="Previous Day Log", lines=5),
        gr.Textbox(value=sample_today_goals, label="Today's Goals", lines=5),
        gr.Textbox(value=sample_time_slots, label="Available Time Slots", lines=4)
    ],
    outputs=gr.Markdown(label="Daily Plan Generated by Agent"),
    title="Self-Reflective Daily Planner Agent",
    description="An LLM-based agent that reflects on unfinished tasks and re-plans the day.",
    submit_btn="Generate Daily Plan",
    clear_btn="Clear Inputs",
    flagging_mode="never"
)
app.launch(share=True)





Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://14b2f0bd166ff87b11.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
